# FinGPT Two-Agent Signal Pipeline — Colab Demo

**Architecture overview**
```
News article
    │
    ▼
Agent 1 (extractor.py)
  ├─ Guided vLLM call  → facts (source, headline, companies, keywords)
  ├─ CoT vLLM call     → chain-of-thought reasoning  (stop at </think>)
  └─ prompt_logprobs   → real log P(POSITIVE|ctx), log P(NEGATIVE|ctx), log P(NEUTRAL|ctx)
         │  softmax(log_probs / CALIBRATION_T)  ← deterministic Python
         ▼
    NewsFingerprint  (sentiment_label, confidence, probabilities)
    │
    ▼
Agent 2 (reasoner.py)
  ├─ CoT vLLM call     → chain-of-thought reasoning  (stop at </think>)
  └─ prompt_logprobs   → real log P(A|ctx), log P(B|ctx), log P(C|ctx)  (A=BUY, B=HOLD, C=SELL)
         │  softmax(log_probs / CALIBRATION_T)  ← deterministic Python
         ▼
    TradingSignal  (direction, confidence, signal_probabilities)
    │
    ▼
Backtest  →  direction_accuracy, Sharpe, total_pnl
```

**Real logits via `prompt_logprobs`:** the LLM is never asked to output numbers.  
Instead, vLLM's `SamplingParams(prompt_logprobs=1)` reads the model's genuine  
token-level log-probabilities for each class label at the decision point.  

**Batch processing:** the backtest feeds vLLM in batches of 10 articles  
(**5 vLLM calls per batch**: guided extraction + sentiment CoT + sentiment scoring  
+ strategy CoT + strategy scoring).  
All probability / confidence values are computed in Python from real model logprobs.

## Cell 1 — GPU check & install

In [ ]:
import subprocess, sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Switch to a GPU runtime (Runtime → Change runtime type → T4/A100).")

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

# Install pipeline dependencies.
# vllm ships its own torch; install it last to avoid conflicts.
!pip install -q datasets transformers accelerate pydantic python-dotenv yfinance tqdm
!pip install -q vllm
print("Dependencies installed.")

## Cell 2 — Mount Drive & clone / update repo

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ── Edit these two lines to match your Drive layout ──────────────────────────
PROJECT_PATH = '/content/drive/MyDrive/FinGPT_Part2'
REPO_URL     = 'https://github.com/juankim834/FinGPT_Project.git'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(os.path.join(PROJECT_PATH, '.git')):
    print(f"Repo found at {PROJECT_PATH} — pulling latest changes.")
    %cd {PROJECT_PATH}
    !git pull
else:
    print(f"Repo not found — cloning into {PROJECT_PATH}.")
    parent = os.path.dirname(PROJECT_PATH)
    os.makedirs(parent, exist_ok=True)
    %cd {parent}
    !git clone {REPO_URL} {os.path.basename(PROJECT_PATH)}
    %cd {PROJECT_PATH}

# Ensure project root is on sys.path so all `from agent1…` imports resolve.
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Shared directories persisted on Drive.
DEMO_OUTPUT_DIR = os.path.join(PROJECT_PATH, 'output')
DIAG_MD_DIR     = os.path.join(DEMO_OUTPUT_DIR, 'diagnostics_md')
for d in [DEMO_OUTPUT_DIR, DIAG_MD_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Project path : {PROJECT_PATH}")
print(f"Output dir   : {DEMO_OUTPUT_DIR}")
print(f"Diagnostics  : {DIAG_MD_DIR}")

## Cell 3 — Load secrets & configure environment

In [ ]:
import os
from google.colab import userdata

# ── vLLM Colab / Jupyter compatibility fix ────────────────────────────────────
# vLLM ≥0.6 (v1 engine) spawns an EngineCore subprocess that calls
# sys.stdout.fileno().  IPython's OutStream has no real fileno(), so the child
# crashes with:  io.UnsupportedOperation: fileno
#
# Setting VLLM_ENABLE_V1_MULTIPROCESSING=0 forces the v1 engine to run
# in-process (InprocClient) — no subprocess spawned, no fileno call.
# Must be set BEFORE LLM() is called (vLLM reads it lazily at init time).
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

# ── News provider ─────────────────────────────────────────────────────────────
os.environ['NEWS_PROVIDER']   = 'finnhub'          # 'finnhub' or 'alpaca'
os.environ['FINNHUB_API_KEY'] = userdata.get('FINNHUB_API_KEY')
os.environ['ALPACA_API_KEY']  = userdata.get('ALPACA_API_KEY')
os.environ['ALPACA_API_SECRET'] = userdata.get('ALPACA_API_SECRET')

# ── Model path ────────────────────────────────────────────────────────────────
# Point to your local merged / quantised model directory on Drive.
# The folder must contain config.json (HuggingFace format).
model_candidates = [
    '/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm',
    '/content/drive/MyDrive/models/DeepSeek-R1-Distill-Llama-8B',
]
resolved_model = next(
    (p for p in model_candidates if os.path.isfile(os.path.join(p, 'config.json'))),
    None,
)
if resolved_model is None:
    raise FileNotFoundError(
        'Could not find a model folder with config.json. '
        f'Checked: {model_candidates}'
    )
os.environ['FINGPT_MODEL_PATH'] = resolved_model

# ── Pipeline settings ─────────────────────────────────────────────────────────
# Share one vLLM engine between Agent 1 and Agent 2 to save VRAM.
os.environ['SHARE_SINGLE_LLM_BETWEEN_AGENTS'] = 'true'

# Calibration temperature applied to softmax(logits / T).
# T > 1 → softer distribution  |  T < 1 → sharper  |  T = 1 → standard softmax
os.environ['FINGPT_CALIBRATION_T']    = '1.2'

# Token budget for CoT + logits generation per article.
os.environ['FINGPT_LOGITS_MAX_TOKENS'] = '1024'

# Debug output directory (raw model outputs saved here for inspection).
os.environ['FINGPT_DIAG_MD_DIR'] = DIAG_MD_DIR

# Persist yfinance price cache across runs.
os.environ['FINGPT_YF_CACHE_PATH'] = os.path.join(DEMO_OUTPUT_DIR, 'yfinance_return_cache.json')

print('Environment configured.')
print(f"  NEWS_PROVIDER            = {os.environ['NEWS_PROVIDER']}")
print(f"  FINGPT_MODEL_PATH        = {os.environ['FINGPT_MODEL_PATH']}")
print(f"  FINGPT_CALIBRATION_T     = {os.environ['FINGPT_CALIBRATION_T']}")
print(f"  FINGPT_LOGITS_MAX_TOKENS = {os.environ['FINGPT_LOGITS_MAX_TOKENS']}")
print(f"  FINGPT_DIAG_MD_DIR       = {os.environ['FINGPT_DIAG_MD_DIR']}")

## Cell 4 — Load vLLM engine (shared between Agent 1 & 2)

In [ ]:
import sys, os, torch
from vllm import LLM, SamplingParams

def _used_vram_gb() -> float:
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

def _gpu_supports_bf16() -> bool:
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability(0)
    return major >= 8  # Ampere+ (A100, A10G) supports fast BF16; T4 does not.

model_id   = os.environ['FINGPT_MODEL_PATH']
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
dtype      = 'bfloat16' if _gpu_supports_bf16() else 'float16'

print(f"Loading model: {model_id}")
print(f"GPU VRAM before load: {_used_vram_gb():.2f} GB / {total_vram:.1f} GB total")
print(f"dtype: {dtype}")

# Belt-and-suspenders stdout fix: if VLLM_ENABLE_V1_MULTIPROCESSING was not
# honoured (cached import), redirect sys.stdout to a real fd before the fork
# so the EngineCore child process can call fileno() without raising.
_nb_stdout = sys.stdout
try:
    sys.stdout = os.fdopen(os.dup(1), 'w', buffering=1)
except Exception:
    pass

# Try progressive fallback profiles if VRAM is tight.
startup_profiles = [
    {'dtype': dtype,      'gpu_memory_utilization': 0.85, 'enforce_eager': True},
    {'dtype': 'float16',  'gpu_memory_utilization': 0.80, 'enforce_eager': True},
]

llm = None
try:
    for i, profile in enumerate(startup_profiles, start=1):
        try:
            print(f"Attempt {i}: {profile}")
            llm = LLM(model=model_id, trust_remote_code=True, disable_log_stats=True, **profile)
            break
        except Exception as exc:
            print(f"  ✗ Attempt {i} failed: {exc}")
finally:
    sys.stdout = _nb_stdout   # restore notebook stdout

if llm is None:
    raise RuntimeError('vLLM failed to start. Check VRAM or model path.')

# Inject shared engine into both agents.
from agent1.extractor import set_shared_vllm_engine as set_a1_engine
from agent2.reasoner  import set_shared_vllm_engine as set_a2_engine
set_a1_engine(llm)
set_a2_engine(llm)

# Warmup.
_ = llm.generate(['Warmup.'], SamplingParams(max_tokens=1, temperature=0.0))

print(f"\nvLLM engine ready.  VRAM after load: {_used_vram_gb():.2f} GB")

## Cell 5 — Smoke test: single article through Agent 1 + Agent 2

Run one article end-to-end before the full backtest so you can inspect the raw logits and probability vectors.

In [ ]:
import json
from agent1.extractor import extract_fingerprint
from agent2.reasoner  import generate_signal

SMOKE_ARTICLE = (
    "News 1\n"
    "Headline: Apple reports record Q4 revenue of $120B, beating Wall Street estimates by 8%.\n"
    "Summary: Apple Inc. (AAPL) delivered blockbuster quarterly results driven by strong iPhone "
    "and services growth. CEO Tim Cook raised full-year guidance and announced a $90B buyback programme."
)

print("── Agent 1: fact extraction + sentiment + event type ──")
fp = extract_fingerprint(SMOKE_ARTICLE)

if fp is None:
    print("  Agent 1 returned None — check diagnostics_md/agent1/ for the raw output.")
else:
    print(f"  Headline        : {fp.headline}")
    print(f"  Companies       : {fp.companies_named}")
    print(f"  Sentiment       : {fp.sentiment_label}  (confidence={fp.sentiment_confidence:.4f})")
    print(f"  Sent logprobs   : {fp.sentiment_logits}")        # [log P(POS), log P(NEG), log P(NEU)]
    print(f"  Sent probs      : {json.dumps(fp.sentiment_probabilities, indent=4)}")
    print(f"  Event type      : {fp.event_type}  (confidence={fp.event_type_confidence})  method={fp.event_type_method}")
    print(f"  Event logprobs  : {fp.event_type_logits}")       # {A: logP, B: logP, ..., G: logP}
    print(f"  Event probs     : {json.dumps(fp.event_type_probabilities, indent=4)}")
    print(f"  Cal. T          : {fp.calibration_T}")

    print("\n── Agent 2: strategy selection ──")
    sig = generate_signal(fp)
    if sig is None:
        print("  Agent 2 returned None — check diagnostics_md/agent2/ and logs/.")
    else:
        print(f"  Direction       : {sig.direction}")
        print(f"  Confidence      : {sig.confidence:.4f}")
        print(f"  Raw logprobs    : {sig.raw_signal_logits}")  # [logP(A), logP(B), logP(C)] pre-PMI
        print(f"  PMI null        : {sig.pmi_null_logprobs}")  # null-context logprobs for PMI correction
        print(f"  PMI alpha       : {sig.pmi_alpha_used}")
        print(f"  PMI-adj logits  : {sig.signal_logits}")      # raw − alpha * null
        print(f"  Signal probs    : {json.dumps(sig.signal_probabilities, indent=4)}")
        print(f"  Filter override : {sig.signal_filter_forced_hold}  ({sig.signal_filter_reason})")
        print(f"  CoT (first 300 chars):\n    {sig.cot[:300]}")

## Cell 6 — Load FinGPT test dataset

In [ ]:
import pandas as pd
from datasets import load_dataset as hf_load_dataset
from backtest.dataset_parser import build_backtest_rows

# ── Dataset source ─────────────────────────────────────────────────────────────
DATASET_SOURCE = 'FinGPT/fingpt-forecaster-dow30-202305-202405'

# Set MAX_ROWS to a small int (e.g. 30) for a quick smoke test; None = full dataset.
MAX_ROWS = None
# ──────────────────────────────────────────────────────────────────────────────

# Load directly from HuggingFace using datasets.load_dataset.
hf_ds = hf_load_dataset(DATASET_SOURCE)
split = next(
    (s for s in ('test', 'validation', 'train') if s in hf_ds),
    next(iter(hf_ds.keys())),
)
print(f"HF dataset  : {DATASET_SOURCE}")
print(f"Split       : '{split}'  ({len(hf_ds[split]):,} rows)")
print(f"Columns     : {hf_ds[split].column_names}")

# Convert to pandas and normalise column names to dataset_parser convention.
df_raw = hf_ds[split].to_pandas()
df_raw.columns = [c.strip().lower() for c in df_raw.columns]

backtest_rows = build_backtest_rows(df_raw)
if MAX_ROWS is not None:
    backtest_rows = backtest_rows[:MAX_ROWS]

print(f"\nBacktest rows after parsing : {len(backtest_rows)}")
print(f"Batch size                  : 10  (5 vLLM call groups per batch)")
print(f"yfinance cache              : {os.environ.get('FINGPT_YF_CACHE_PATH')}")

preview = pd.DataFrame([
    {
        'ticker'      : r['ticker'],
        'start_date'  : r['start_date'],
        'end_date'    : r['end_date'],
        'fingpt_label': r['fingpt_label'],
        'article_text': r['article_text'][:80] + '…',
    }
    for r in backtest_rows
])
display(preview.head(10))

## Cell 7 — Run backtest (batched inference)

The backtester groups rows into batches of 10.  For each batch it makes **5 vLLM calls**:

| # | Call | Purpose |
|---|------|---------|
| 1 | Guided extraction | Facts (source, headline, companies, keywords) for all 10 articles |
| 2 | Sentiment CoT | Chain-of-thought reasoning, stopped at `</think>`, for all 10 articles |
| 3 | Sentiment scoring | `prompt_logprobs` for 10 × 3 = 30 scoring prompts → real log P(class\|ctx) |
| 4 | Strategy CoT | Chain-of-thought reasoning for the valid fingerprints in the batch |
| 5 | Strategy scoring | `prompt_logprobs` for N × 3 scoring prompts → real log P(strategy\|ctx) |

Price fetching and result assembly happen row-by-row after inference.

In [ ]:
from datetime import datetime, timezone
from backtest.backtester import run_backtest
from backtest.price_fetcher import get_cache_stats

ts               = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
backtest_outpath = os.path.join(DEMO_OUTPUT_DIR, f'backtest_{ts}.csv')

results_df = run_backtest(
    dataset_path=DATASET_SOURCE,
    output_path=backtest_outpath,
    max_rows=MAX_ROWS,
)

successful  = results_df[results_df['skipped_reason'] == '']
cache_stats = get_cache_stats()

print(f"Saved        : {backtest_outpath}")
print(f"Total rows   : {len(results_df)}")
print(f"Successful   : {len(successful)}")
print(f"yfinance cache (disk / memory): {cache_stats['disk_entries']} / {cache_stats['memory_entries']}")
print(f"\nColumns in CSV ({len(results_df.columns)}): {list(results_df.columns)}")

# ── Agent 1 sentiment columns ──────────────────────────────────────────────────
sent_cols = [c for c in [
    'ticker', 'fingpt_label',
    'sentiment_label', 'sentiment_confidence',
    'sentiment_logprob_POSITIVE', 'sentiment_logprob_NEGATIVE', 'sentiment_logprob_NEUTRAL',
    'sentiment_prob_POSITIVE',    'sentiment_prob_NEGATIVE',    'sentiment_prob_NEUTRAL',
] if c in results_df.columns]
print('\nAgent 1 — sentiment (first 5 rows):')
display(results_df[sent_cols].head(5))

# ── Agent 1 event-type columns ─────────────────────────────────────────────────
event_cols = [c for c in [
    'ticker', 'event_type', 'event_type_confidence', 'event_type_margin', 'event_type_method',
    'event_logprob_A', 'event_logprob_B', 'event_logprob_C', 'event_logprob_D',
    'event_logprob_E', 'event_logprob_F', 'event_logprob_G',
    'event_prob_A',    'event_prob_B',    'event_prob_C',    'event_prob_D',
    'event_prob_E',    'event_prob_F',    'event_prob_G',
] if c in results_df.columns]
print('\nAgent 1 — event type (first 5 rows):')
display(results_df[event_cols].head(5))

# ── Agent 2 signal + PMI columns ──────────────────────────────────────────────
signal_cols = [c for c in [
    'ticker', 'direction', 'confidence',
    'raw_signal_logprob_A', 'raw_signal_logprob_B', 'raw_signal_logprob_C',
    'pmi_null_logprob_A',   'pmi_null_logprob_B',   'pmi_null_logprob_C',
    'pmi_adjusted_logit_A', 'pmi_adjusted_logit_B', 'pmi_adjusted_logit_C',
    'signal_prob_A', 'signal_prob_B', 'signal_prob_C',
    'pmi_alpha_used', 'calibration_T',
    'signal_filter_forced_hold', 'signal_filter_reason',
    'realized_return', 'strategy_return', 'skipped_reason',
] if c in results_df.columns]
print('\nAgent 2 — signal + PMI (first 5 rows):')
display(results_df[signal_cols].head(5))

## Cell 8 — Backtest metrics

In [ ]:
import json
from backtest.backtester import compute_metrics

metrics = compute_metrics(results_df)

# Average sentiment confidence.
if 'sentiment_confidence' in results_df.columns:
    v = results_df['sentiment_confidence'].dropna().astype(float)
    metrics['avg_sentiment_confidence'] = float(v.mean()) if not v.empty else 0.0

# Average signal confidence (primary column is 'confidence'; fall back to legacy alias).
conf_col = 'confidence' if 'confidence' in results_df.columns else 'signal_confidence'
if conf_col in results_df.columns:
    v = results_df[conf_col].dropna().astype(float)
    metrics['avg_signal_confidence'] = float(v.mean()) if not v.empty else 0.0

# Average PMI alpha actually used (useful to confirm config was applied).
if 'pmi_alpha_used' in results_df.columns:
    v = results_df['pmi_alpha_used'].dropna().astype(float)
    metrics['avg_pmi_alpha_used'] = float(v.mean()) if not v.empty else 0.0

# Average calibration temperature.
if 'calibration_T' in results_df.columns:
    v = results_df['calibration_T'].dropna().astype(float)
    metrics['avg_calibration_T'] = float(v.mean()) if not v.empty else 0.0

print('=' * 52)
print('  Backtest Metrics')
print('=' * 52)
scalar_items = {k: v for k, v in metrics.items() if not isinstance(v, dict)}
nested_items = {k: v for k, v in metrics.items() if isinstance(v, dict)}
for k, v in scalar_items.items():
    if isinstance(v, float):
        print(f'  {k:<38} {v:.4f}')
    else:
        print(f'  {k:<38} {v}')
for section, breakdown in nested_items.items():
    print(f'\n  {section}:')
    for sub_k, sub_v in breakdown.items():
        print(f'    {sub_k:<30} {sub_v}')
print('=' * 52)

metrics_path = backtest_outpath.replace('.csv', '_metrics.json')
with open(metrics_path, 'w') as fh:
    json.dump(metrics, fh, indent=2)
print(f"\nMetrics saved to: {metrics_path}")

## Cell 9 — Skip analysis & logit distribution

In [ ]:
# ── Skip breakdown ─────────────────────────────────────────────────────────────
skip_counts = results_df['skipped_reason'].value_counts(dropna=False)
print('Skip breakdown:')
display(skip_counts.to_frame(name='count'))

# ── Sentiment distribution ─────────────────────────────────────────────────────
if 'sentiment_label' in results_df.columns:
    print('\nSentiment label distribution (Agent 1):')
    display(results_df['sentiment_label'].value_counts(dropna=True).to_frame(name='count'))

# ── Event type distribution ────────────────────────────────────────────────────
if 'event_type' in results_df.columns:
    print('\nEvent type distribution (Agent 1):')
    display(results_df['event_type'].value_counts(dropna=True).to_frame(name='count'))

# ── Event type method distribution (shows abstention rate) ────────────────────
if 'event_type_method' in results_df.columns:
    print('\nEvent type method (Agent 1):')
    display(results_df['event_type_method'].value_counts(dropna=True).to_frame(name='count'))

# ── Signal direction distribution (primary 'direction' col) ───────────────────
dir_col = 'direction' if 'direction' in results_df.columns else 'signal_direction'
if dir_col in results_df.columns:
    print(f'\nSignal direction distribution (Agent 2, col={dir_col!r}):')
    display(results_df[dir_col].value_counts(dropna=True).to_frame(name='count'))

# ── Signal filter forced-hold breakdown ───────────────────────────────────────
if 'signal_filter_forced_hold' in results_df.columns:
    ok = results_df[results_df['skipped_reason'] == '']
    print('\nSignal filter forced-hold (Agent 2):')
    display(ok['signal_filter_forced_hold'].value_counts(dropna=False).to_frame(name='count'))
    if ok['signal_filter_forced_hold'].any():
        print('  Forced-hold reasons:')
        display(ok['signal_filter_reason'].value_counts(dropna=True).to_frame(name='count'))

# ── Sample: all raw logprob columns for first 5 successful rows ────────────────
print('\nSample — all raw logprobs / PMI / probabilities (first 5 successful rows):')
ok = results_df[results_df['skipped_reason'] == '']
sample_cols = [c for c in [
    'ticker',
    'sentiment_label', 'sentiment_confidence',
    'sentiment_logprob_POSITIVE', 'sentiment_logprob_NEGATIVE', 'sentiment_logprob_NEUTRAL',
    'event_type', 'event_type_confidence', 'event_type_method',
    'event_logprob_A', 'event_logprob_B', 'event_logprob_C', 'event_logprob_D',
    'event_logprob_E', 'event_logprob_F', 'event_logprob_G',
    'direction', 'confidence',
    'raw_signal_logprob_A', 'raw_signal_logprob_B', 'raw_signal_logprob_C',
    'pmi_null_logprob_A',   'pmi_null_logprob_B',   'pmi_null_logprob_C',
    'pmi_adjusted_logit_A', 'pmi_adjusted_logit_B', 'pmi_adjusted_logit_C',
    'signal_prob_A', 'signal_prob_B', 'signal_prob_C',
    'pmi_alpha_used', 'calibration_T',
] if c in results_df.columns]
display(ok[sample_cols].head(5))

## Cell 10 — (Optional) Batch smoke test: extract_fingerprint_batch + generate_signal_batch

Directly exercises the batch API on a small set of articles.

In [ ]:
from agent1.extractor import extract_fingerprint_batch
from agent2.reasoner  import generate_signal_batch

BATCH_ARTICLES = [
    (
        'News 1\nHeadline: Microsoft Azure revenue surges 28% on AI demand.\n'
        'Summary: Microsoft reported cloud revenue well above estimates, driven by '
        'Copilot adoption across enterprise customers.'
    ),
    (
        'News 1\nHeadline: Tesla misses Q3 deliveries by 5% amid demand slowdown.\n'
        'Summary: Tesla delivered 435,000 vehicles in Q3, below analyst expectations '
        'of 457,000, citing price competition and weaker European demand.'
    ),
    (
        'News 1\nHeadline: Fed holds rates steady, signals one cut in 2025.\n'
        'Summary: The Federal Reserve kept its benchmark rate unchanged at 5.25–5.50% '
        'and updated its dot plot to show one 25bp cut expected later in the year.'
    ),
]

print(f"Running batch extraction on {len(BATCH_ARTICLES)} articles…")
fingerprints = extract_fingerprint_batch(BATCH_ARTICLES)

print(f"Running batch signal generation…")
valid_fps = [fp for fp in fingerprints if fp is not None]
signals   = generate_signal_batch(valid_fps) if valid_fps else []

sig_iter = iter(signals)
for i, fp in enumerate(fingerprints):
    print(f"\n─── Article {i+1} ───")
    if fp is None:
        print('  Agent 1: FAILED')
        continue
    print(f"  Headline        : {fp.headline}")
    print(f"  Companies       : {fp.companies_named}")
    print(f"  Sentiment       : {fp.sentiment_label}  conf={fp.sentiment_confidence:.4f}")
    print(f"  Sent logprobs   : {fp.sentiment_logits}")       # [log P(POS), log P(NEG), log P(NEU)]
    print(f"  Event type      : {fp.event_type}  conf={fp.event_type_confidence}  method={fp.event_type_method}")
    print(f"  Event logprobs  : {fp.event_type_logits}")      # {A: logP, ..., G: logP}
    print(f"  Event probs     : {fp.event_type_probabilities}")
    sig = next(sig_iter, None)
    if sig is None:
        print('  Agent 2: FAILED')
    else:
        print(f"  Direction       : {sig.direction}  conf={sig.confidence:.4f}")
        print(f"  Raw logprobs    : {sig.raw_signal_logits}")  # [logP(A), logP(B), logP(C)] pre-PMI
        print(f"  PMI null        : {sig.pmi_null_logprobs}")
        print(f"  PMI alpha       : {sig.pmi_alpha_used}")
        print(f"  PMI-adj logits  : {sig.signal_logits}")      # raw − alpha * null
        print(f"  Signal probs    : {sig.signal_probabilities}")
        print(f"  Filter override : {sig.signal_filter_forced_hold}  ({sig.signal_filter_reason})")

## Cell 11 — Release GPU runtime

In [ ]:
from google.colab import runtime
runtime.unassign()